# Assignment 3 — Data preprocessing: doing it in the right order

**Dataset:** IBM Telco Customer Churn (7,043 rows, 20 feature columns, target = `Churn`)
**Source link:** https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv

Why this dataset fits the brief: more than 3,000 rows, a mix of numeric and categorical columns, missing values (`TotalCharges` has blank entries), and an imbalanced target (about 26% churn).

The notebook splits first, and every imputer, encoder and scaler is fitted on the training set only.

In [ ]:
!pip install -q imbalanced-learn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import (OneHotEncoder, OrdinalEncoder, StandardScaler,
                                   MinMaxScaler, RobustScaler, PowerTransformer)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import joblib

RS = 42
np.random.seed(RS)

## Load the data

In [ ]:
URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
try:
    raw = pd.read_csv(URL)
except Exception:
    # fallback: upload Telco-Customer-Churn.csv with the Colab file panel
    raw = pd.read_csv("Telco-Customer-Churn.csv")

# TotalCharges holds blank strings for some rows -> make it numeric, blanks become NaN.
# This is type conversion only (nothing is fitted), so it is safe before the split.
raw["TotalCharges"] = pd.to_numeric(raw["TotalCharges"], errors="coerce")

df = raw.drop(columns="customerID")          # an ID carries no signal
y_all = (df["Churn"] == "Yes").astype(int)   # 1 = churned
X_all = df.drop(columns="Churn")

print("Shape:", raw.shape)
print("\nMissing values per column:")
print(raw.isna().sum()[raw.isna().sum() > 0])
print("\nTarget distribution:")
print(y_all.value_counts())

## Q1. Stratified train/test split FIRST

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, stratify=y_all, random_state=RS)

print("Train:", X_train.shape, " Test:", X_test.shape)
print("Churn rate  full: %.4f | train: %.4f | test: %.4f" % (y_all.mean(), y_train.mean(), y_test.mean()))

**Why split first:** every statistic we learn from data (means, medians, category lists, scaling ranges) must come from the training rows only. If we compute them before splitting, information from the test rows leaks into training and the test score stops being an honest estimate of performance on new data.
Stratifying on the target keeps the churn ratio the same in both parts, which matters because the classes are imbalanced.

## Q2. Three ways to handle missing values (fitted on train only)

In [ ]:
num_cols = ["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges"]
col = "TotalCharges"
print("Missing in train:", X_train[col].isna().sum(), "| in test:", X_test[col].isna().sum())
print("Tenure of the rows with missing TotalCharges:", X_train.loc[X_train[col].isna(), "tenure"].unique())

# (a) drop the rows
s_drop = X_train.dropna(subset=[col])[col]

# (b) fill with the median (learned from train)
med_imp = SimpleImputer(strategy="median").fit(X_train[[col]])
s_med = pd.Series(med_imp.transform(X_train[[col]]).ravel(), index=X_train.index)

# (c) KNNImputer on the numeric columns (neighbours learned from train)
knn_imp = KNNImputer(n_neighbors=5).fit(X_train[num_cols])
knn_train = pd.DataFrame(knn_imp.transform(X_train[num_cols]), columns=num_cols, index=X_train.index)
s_knn = knn_train[col]

# test set: transform only, never fit
test_med = med_imp.transform(X_test[[col]])
test_knn = knn_imp.transform(X_test[num_cols])

summary = pd.DataFrame({
    "rows": [len(s_drop), len(s_med), len(s_knn)],
    "mean": [s_drop.mean(), s_med.mean(), s_knn.mean()],
    "std":  [s_drop.std(),  s_med.std(),  s_knn.std()],
}, index=["drop rows", "median fill", "KNNImputer"])
print(summary.round(2))

**Which I would keep: `KNNImputer`.** Dropping loses rows, and the missing rows are not random (they are brand-new customers with tenure 0), so dropping removes a whole kind of customer and the test set cannot be dropped at prediction time anyway. The median fill ignores everything else we know about the customer. KNNImputer uses `tenure` and `MonthlyCharges` to estimate a sensible value (small for new customers) while keeping the column's mean and spread close to the original.

## Q3. Outliers: IQR rule, z-score rule, percentile capping

In [ ]:
s = s_med   # TotalCharges, imputed, train only

q1, q3 = s.quantile(0.25), s.quantile(0.75)
iqr = q3 - q1
lo_iqr, hi_iqr = q1 - 1.5 * iqr, q3 + 1.5 * iqr
flag_iqr = (s < lo_iqr) | (s > hi_iqr)

mu, sd = s.mean(), s.std()
flag_z = ((s - mu) / sd).abs() > 3

p1, p99 = s.quantile(0.01), s.quantile(0.99)
flag_cap = (s < p1) | (s > p99)
s_capped = s.clip(p1, p99)

A, B, C = set(s.index[flag_iqr]), set(s.index[flag_z]), set(s.index[flag_cap])
print("Rows flagged  IQR: %d | z-score: %d | 1st/99th percentile: %d" % (len(A), len(B), len(C)))
print("Overlap IQR & z-score:", len(A & B))
print("Overlap IQR & percentile:", len(A & C))
print("Overlap z-score & percentile:", len(B & C))
print("Flagged by all three:", len(A & B & C))

# same IQR check on the other numeric columns, for context (loop over column names is fine)
for c in ["tenure", "MonthlyCharges"]:
    x = X_train[c]
    a, b = x.quantile(0.25), x.quantile(0.75)
    print(c, "-> IQR flags:", int(((x < a - 1.5*(b-a)) | (x > b + 1.5*(b-a))).sum()))

**Chosen method: capping at the 1st and 99th percentile.** It keeps every row, needs no assumption about the shape of the distribution (the z-score rule assumes a roughly symmetric column, and `TotalCharges` is right-skewed), and unlike the IQR rule it always limits the extreme tails. High totals here come from genuine long-tenure customers, not data errors, so deleting those rows would be wrong; limiting their influence is enough.

## Q4. Encoding: one-hot, ordinal, frequency

In [ ]:
d = X_train.copy()
print("Columns at start:", d.shape[1])

# one-hot: low-cardinality column
ohe_demo = OneHotEncoder(sparse_output=False, handle_unknown="ignore").fit(d[["InternetService"]])
oh = pd.DataFrame(ohe_demo.transform(d[["InternetService"]]),
                  columns=ohe_demo.get_feature_names_out(["InternetService"]), index=d.index)
d = pd.concat([d.drop(columns="InternetService"), oh], axis=1)
print("After one-hot (InternetService, %d categories): %d columns" % (len(ohe_demo.categories_[0]), d.shape[1]))

# ordinal: ordered column with an explicit order
contract_order = ["Month-to-month", "One year", "Two year"]
ord_enc = OrdinalEncoder(categories=[contract_order], handle_unknown="use_encoded_value",
                         unknown_value=-1).fit(d[["Contract"]])
d["Contract"] = ord_enc.transform(d[["Contract"]]).ravel()
print("After ordinal (Contract): %d columns" % d.shape[1])

# frequency: category -> share of training rows (the highest-cardinality column in this dataset)
freq_map = X_train["PaymentMethod"].value_counts(normalize=True)
d["PaymentMethod"] = d["PaymentMethod"].map(freq_map)
print("After frequency encoding (PaymentMethod): %d columns" % d.shape[1])
print(freq_map.round(4))

## Q5. A category in test that never appeared in train

In [ ]:
new = X_train.head(3).copy()
new["InternetService"] = "Satellite"      # never seen in train
new["Contract"] = "Three year"            # never seen in train
new["PaymentMethod"] = "Bitcoin"          # never seen in train

# Without protection the default encoder crashes:
try:
    OneHotEncoder().fit(X_train[["InternetService"]]).transform(new[["InternetService"]])
except ValueError as e:
    print("Default OneHotEncoder crashed ->", str(e)[:80], "...")

# With protection (handle_unknown settings above) nothing crashes:
print("\nOne-hot of unseen category (all zeros):")
print(ohe_demo.transform(new[["InternetService"]]))
print("Ordinal of unseen category (-1):", ord_enc.transform(new[["Contract"]]).ravel())
print("Frequency of unseen category (0.0):", new["PaymentMethod"].map(freq_map).fillna(0.0).values)

## Q6. Scaling a skewed column: Standard, MinMax, Robust

In [ ]:
x = s_med.values.reshape(-1, 1)    # TotalCharges, train only
scaled = {
    "Original": x.ravel(),
    "StandardScaler": StandardScaler().fit(x).transform(x).ravel(),
    "MinMaxScaler": MinMaxScaler().fit(x).transform(x).ravel(),
    "RobustScaler": RobustScaler().fit(x).transform(x).ravel(),
}
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, (name, vals) in zip(axes, scaled.items()):
    ax.hist(vals, bins=40)
    ax.set_title(name)
plt.tight_layout()
plt.show()

**Best with outliers: `RobustScaler`.** It centres on the median and scales by the IQR, so a few extreme values do not stretch the scale the way they do for the mean/std (StandardScaler) or the min/max (MinMaxScaler).

## Q7. Skewness: log1p and Yeo-Johnson

In [ ]:
col_s = s_med
print("Skewness original: %.3f" % col_s.skew())

log_vals = np.log1p(col_s)
print("Skewness after log1p: %.3f" % log_vals.skew())

pt = PowerTransformer(method="yeo-johnson").fit(col_s.values.reshape(-1, 1))
yj_vals = pd.Series(pt.transform(col_s.values.reshape(-1, 1)).ravel())
print("Skewness after Yeo-Johnson: %.3f" % yj_vals.skew())

## Q8. ColumnTransformer inside a Pipeline (impute + encode + scale)

In [ ]:
ord_cols = ["Contract"]
nom_cols = [c for c in X_train.columns if c not in num_cols + ord_cols]

numeric_pipe = Pipeline([("impute", KNNImputer(n_neighbors=5)),
                         ("scale", StandardScaler())])
ordinal_pipe = Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                         ("encode", OrdinalEncoder(categories=[contract_order],
                                                   handle_unknown="use_encoded_value", unknown_value=-1)),
                         ("scale", StandardScaler())])
nominal_pipe = Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                         ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])

ct = ColumnTransformer([("num", numeric_pipe, num_cols),
                        ("ord", ordinal_pipe, ord_cols),
                        ("nom", nominal_pipe, nom_cols)],
                       sparse_threshold=0)
preprocess = Pipeline([("columntransformer", ct)])

Xtr_p = preprocess.fit_transform(X_train)   # fit on train only
Xte_p = preprocess.transform(X_test)        # transform only

print("Train output shape:", Xtr_p.shape)
print("Test  output shape:", Xte_p.shape)
assert Xtr_p.shape[1] == Xte_p.shape[1], "column counts must match"
assert not np.isnan(Xtr_p).any() and not np.isnan(Xte_p).any()
print("Column counts match and no NaN remains.")

## Q9. Demonstrating leakage (the one place where it is the point)

In [ ]:
feat = "MonthlyCharges"

# WRONG: scaler fitted on the full dataset (train + test rows together)
wrong_scaler = StandardScaler().fit(df[[feat]])
wrong = wrong_scaler.transform(X_test[[feat]]).ravel()

# RIGHT: scaler fitted on the training rows only
right_scaler = StandardScaler().fit(X_train[[feat]])
right = right_scaler.transform(X_test[[feat]]).ravel()

print("First 5 scaled test values")
print("  leaky  :", np.round(wrong[:5], 6))
print("  correct:", np.round(right[:5], 6))
print("  difference:", np.round(wrong[:5] - right[:5], 6))
print("Mean absolute difference over the whole test set: %.6f" % np.abs(wrong - right).mean())
print("Mean/std used - leaky: %.4f / %.4f | correct: %.4f / %.4f" % (
    wrong_scaler.mean_[0], wrong_scaler.scale_[0], right_scaler.mean_[0], right_scaler.scale_[0]))

**Why the first version makes the test score a lie:**
1. The leaky scaler used the mean and standard deviation of the test rows too, so the test data helped shape its own preprocessing.
2. The model is then evaluated on data that was already "seen" in the pipeline, which makes the test rows look slightly more familiar than real, unseen customers.
3. The reported score is therefore optimistic and will not hold up in production, where new data can never contribute to the scaler.

## Q10. Class ratio and SMOTE (training set only)

In [ ]:
from imblearn.over_sampling import SMOTE

counts = Counter(y_train)
print("Train class counts before SMOTE:", dict(counts))
print("Class ratio (majority : minority) = %.2f : 1" % (counts[0] / counts[1]))

smote = SMOTE(random_state=RS)
Xtr_res, ytr_res = smote.fit_resample(Xtr_p, y_train)
print("Train class counts after SMOTE: ", dict(Counter(ytr_res)))
print("Test class counts (untouched):  ", dict(Counter(y_test)))

**Why the test set is left untouched:** the test set must look like real future data, which is imbalanced. SMOTE creates synthetic minority rows from training neighbours; adding synthetic rows to the test set would make the evaluation easier and unrealistic, and would also leak training information into it.

## Q11. Save the pipeline and use it on one raw row

In [ ]:
joblib.dump(preprocess, "pipeline.joblib")
print("Saved pipeline.joblib")

try:
    from google.colab import files
    files.download("pipeline.joblib")
except ImportError:
    print("Not running in Colab - the file is in the current folder.")

In [ ]:
# Fresh load + one new row of RAW data, end to end
loaded = joblib.load("pipeline.joblib")

new_row = pd.DataFrame([{
    "gender": "Female", "SeniorCitizen": 0, "Partner": "Yes", "Dependents": "No",
    "tenure": 12, "PhoneService": "Yes", "MultipleLines": "No",
    "InternetService": "Fiber optic", "OnlineSecurity": "No", "OnlineBackup": "Yes",
    "DeviceProtection": "No", "TechSupport": "No", "StreamingTV": "Yes",
    "StreamingMovies": "No", "Contract": "Month-to-month", "PaperlessBilling": "Yes",
    "PaymentMethod": "Electronic check", "MonthlyCharges": 85.5, "TotalCharges": 1020.0,
}])

out = loaded.transform(new_row)
print("Transformed shape:", out.shape)
assert out.shape[1] == Xtr_p.shape[1]
print(out.round(3))

## Q12. My choices and reasoning

- **Q2 (missing values): `KNNImputer`.** Dropping rows removes a specific group (new customers with tenure 0) and cannot be done at prediction time; the median ignores all other columns; KNN uses `tenure` and `MonthlyCharges` to give a sensible estimate.
- **Q3 (outliers): capping at the 1st and 99th percentile.** It keeps all rows, makes no symmetry assumption on a right-skewed column, and the large totals are real long-tenure customers, so limiting their influence is better than deleting them.
- **Q6 (scaling): `RobustScaler`.** It uses the median and IQR, so extreme values do not distort the scale the way they do for the mean/std or the min/max.